# Residual patterns

If the straight line is the right mean structure, the residuals should have no systematic shape. A curve left in the residuals, or a spread that grows with $x$, is visible in a small table without a plot: the signs and the squared sizes already have a pattern.


## A parabola fitted by a line

Let $x = 1, 2, 3, 4, 5$ and $y = x^2$, so $y = 1, 4, 9, 16, 25$. Then $\bar x = 3$, $\bar y = 11$,

$$
\begin{aligned}
S_{xx} &= 10, \\
S_{xy} &= (-2)(-10) + (-1)(-7) + 0 + (1)(5) + (2)(14) = 20 + 7 + 5 + 28 = 60, \\
b_1 &= 6, \qquad b_0 = 11 - 18 = -7.
\end{aligned}
$$

The predictions $-1, 5, 11, 17, 23$ leave residuals

$$
e = 2,\ -1,\ -2,\ -1,\ 2.
$$

The signs are $+,\ -,\ -,\ -,\ +$. That is the parabola showing through the line: the line is too low at the ends and too high in the middle. The residual sum of squares is $4 + 1 + 4 + 1 + 4 = 14$.

Add the column $x^2$. Then $y = 0 + 0\cdot x + 1\cdot x^2$ fits every row, and the residual sum of squares is zero. The pattern in the signs was the missing column.


In [1]:
# A straight line through a square leaves a U-shaped residual. The square removes it.
import numpy as np
x = np.array([1.0, 2, 3, 4, 5])
y = x ** 2
line = np.column_stack([np.ones(5), x])
beta = np.linalg.solve(line.T @ line, line.T @ y)
residuals = y - line @ beta
quadratic = np.column_stack([line, x ** 2])
beta_q = np.linalg.solve(quadratic.T @ quadratic, quadratic.T @ y)
print("line", beta)
print("residuals", residuals)
print("quadratic", beta_q, "RSS", np.sum((y - quadratic @ beta_q) ** 2))
assert np.allclose(beta, [-7, 6])
assert np.allclose(residuals, [2, -1, -2, -1, 2])
assert abs(np.sum(residuals ** 2) - 14) < 1e-9
assert np.allclose(beta_q, [0, 0, 1])


line [-7.  6.]
residuals [ 2. -1. -2. -1.  2.]
quadratic [-8.2308763e-14  6.5528535e-14  1.0000000e+00] RSS 1.576057806970073e-27


## Spread that grows with $x$

A different failure leaves the mean line correct and makes the scatter unequal. Take three copies of $x = 1$ with response $2$, and three copies of $x = 5$ with responses $6, 10, 14$. Then $\bar x = 3$, $\bar y = 6$, $S_{xx} = 24$, and $S_{xy} = 48$, so the least-squares line is $\widehat y = 2x$. The residuals are

$$
0,\ 0,\ 0,\ -4,\ 0,\ 4.
$$

They sum to zero and they are orthogonal to $x$, so the normal equations are satisfied. Their squares are not comparable in the two groups: the average squared residual at $x = 1$ is $0$, and at $x = 5$ it is

$$
\dfrac{16 + 0 + 16}{3} = \dfrac{32}{3}.
$$

The assumption $\operatorname{Var}(\varepsilon) = \sigma^2 I$ asked for one variance. This table has two. Ordinary least squares still hits the mean line because the group-mean residuals are zero. The standard errors from $s^2(X^{\mathsf T}X)^{-1}$ do not describe the noise that was actually present.


In [2]:
# Equal mean residuals can hide unequal residual spreads.
import numpy as np
x = np.array([1.0, 1, 1, 5, 5, 5])
y = np.array([2.0, 2, 2, 6, 10, 14])
X = np.column_stack([np.ones(6), x])
beta = np.linalg.solve(X.T @ X, X.T @ y)
residuals = y - X @ beta
left = residuals[:3]
right = residuals[3:]
print("beta", beta)
print("residuals", residuals)
print("mean squared residual by group", np.mean(left ** 2), np.mean(right ** 2))
assert np.allclose(beta, [0, 2])
assert np.allclose(X.T @ residuals, 0)
assert abs(np.mean(left ** 2)) < 1e-9
assert abs(np.mean(right ** 2) - 32 / 3) < 1e-9


beta [1.92438658e-15 2.00000000e+00]
residuals [-1.33226763e-15 -1.33226763e-15 -1.33226763e-15 -4.00000000e+00
  0.00000000e+00  4.00000000e+00]
mean squared residual by group 1.7749370367472766e-30 10.666666666666666


## One change: the same spreads, the wrong line

If the right-hand responses are shifted to $8, 12, 16$, the group spreads stay the same and the line changes. The new right-hand mean is $12$ rather than $10$, so the slope that joins $(1, 2)$ to $(5, 12)$ is $10/4 = 5/2$, and the intercept is $2 - 5/2 = -1/2$. Residual shape and residual location answer different questions. A U-shaped sign pattern is about the mean. A growing square is about the variance.


In [3]:
# Shifting one group changes the slope and keeps that group's spread.
import numpy as np
x = np.array([1.0, 1, 1, 5, 5, 5])
y = np.array([2.0, 2, 2, 8, 12, 16])
X = np.column_stack([np.ones(6), x])
beta = np.linalg.solve(X.T @ X, X.T @ y)
residuals = y - X @ beta
print("beta", beta)
print("right-hand residuals", residuals[3:])
assert np.allclose(beta, [-0.5, 2.5])
assert np.allclose(sorted(residuals[3:]), [-4, 0, 4])


beta [-0.5  2.5]
right-hand residuals [-4.  0.  4.]


## Pitfall

Residuals that sum to zero are not residuals that have no pattern. Both examples in this lesson satisfy $X^{\mathsf T}e = 0$. The normal equations can be perfect while the model is wrong: they only say that the chosen columns were used optimally. They do not say that the right columns were chosen, and they do not say that one variance fits every row.


In [4]:
# Orthogonality held for the curved fit and for the unequal-spread fit.
import numpy as np
x = np.array([1.0, 2, 3, 4, 5])
y = x ** 2
X = np.column_stack([np.ones(5), x])
residuals = y - X @ np.linalg.solve(X.T @ X, X.T @ y)
print("sum of residuals", residuals.sum())
print("signs", np.sign(residuals).astype(int).tolist())
assert abs(residuals.sum()) < 1e-12
assert np.sign(residuals).tolist() == [1, -1, -1, -1, 1]


sum of residuals 2.6645352591003757e-15
signs [1, -1, -1, -1, 1]


## Summary

- The line $-7 + 6x$ through $y = x^2$ leaves residuals $2, -1, -2, -1, 2$. The column $x^2$ removes them.
- Residuals $0, 0, 0, -4, 0, 4$ satisfy the normal equations and have mean squared size $0$ at $x = 1$ and $32/3$ at $x = 5$.
- $X^{\mathsf T}e = 0$ means the columns were fitted. It does not mean the model is adequate.
